In [1]:
import torch
import pandas as pd
import math
from collections import Counter
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

# =========================
# 1. Load fine-tuned GPT-2
# =========================
model_path = "/kaggle/input/gpt2-trained-variants/pytorch/default/9/gpt2-large-hp-finetuned/gpt2-large-hp-finetuned"  # <-- your GPT2 fine-tuned path
tokenizer = GPT2TokenizerFast.from_pretrained(model_path)
model = GPT2LMHeadModel.from_pretrained(model_path).to("cuda")
model.eval()

# =========================
# 2. Perplexity Function
# =========================
def calculate_perplexity(sentence):
    if not sentence.strip():
        return float("nan")
    encodings = tokenizer(sentence, return_tensors="pt", truncation=True, max_length=128)
    input_ids = encodings.input_ids.to("cuda")
    with torch.no_grad():
        outputs = model(input_ids, labels=input_ids)
        loss = outputs.loss
    return math.exp(loss.item())

# =========================
# 3. Diversity Functions
# =========================
def ttr(text):
    tokens = text.lower().split()
    return len(set(tokens)) / len(tokens) if tokens else 0

def distinct_n(texts, n=1):
    total_ngrams = 0
    unique_ngrams = set()
    for text in texts:
        tokens = text.lower().split()
        if len(tokens) < n:
            continue
        ngrams = [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
        total_ngrams += len(ngrams)
        unique_ngrams.update(ngrams)
    return len(unique_ngrams) / total_ngrams if total_ngrams > 0 else 0

def entropy(texts):
    tokens = []
    for text in texts:
        tokens.extend(text.lower().split())
    counter = Counter(tokens)
    total = sum(counter.values())
    probs = [count/total for count in counter.values()]
    return -sum(p * math.log2(p) for p in probs if p > 0)

# =========================
# 4. Load CSV
# =========================
csv_path = "/kaggle/input/hp-all/Final_results.csv"  # <-- replace with your file
df = pd.read_csv(csv_path)

caption_cols = ["blip_base_raw", "blip_large_raw", "blip_base_fine_tune",
                "blip_large_fine_tune", "bart_lora", "flan_aug"]

# =========================
# 5. Compute Per-Caption Perplexity
# =========================
for col in caption_cols:
    print(f"Computing perplexity for {col}...")
    df[f"{col}_ppl"] = df[col].astype(str).apply(calculate_perplexity)

# =========================
# 6. Compute Aggregate Diversity Metrics
# =========================
diversity_results = {}
for col in caption_cols:
    texts = df[col].dropna().astype(str).tolist()
    diversity_results[col] = {
        "avg_ppl": df[f"{col}_ppl"].mean(),
        "ttr": sum(ttr(txt) for txt in texts) / len(texts),
        "distinct1": distinct_n(texts, n=1),
        "distinct2": distinct_n(texts, n=2),
        "entropy": entropy(texts)
    }

# =========================
# 7. Save New CSV
# =========================
output_path = "/kaggle/working/captions_with_metrics.csv"
df.to_csv(output_path, index=False)
print(f"✅ Saved per-caption metrics to {output_path}")

# =========================
# 8. Print Aggregate Results
# =========================
print("\n📊 Aggregate Diversity & Fluency Metrics (per model):")
agg_df = pd.DataFrame(diversity_results).T
print(agg_df)

# Save aggregate results too
agg_df.to_csv("/kaggle/working/diversity_summary.csv")

2025-10-05 16:58:50.685186: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759683530.861701      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759683530.921233      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Computing perplexity for blip_base_raw...


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Computing perplexity for blip_large_raw...
Computing perplexity for blip_base_fine_tune...
Computing perplexity for blip_large_fine_tune...
Computing perplexity for bart_lora...
Computing perplexity for flan_aug...
✅ Saved per-caption metrics to /kaggle/working/captions_with_metrics.csv

📊 Aggregate Diversity & Fluency Metrics (per model):
                          avg_ppl       ttr  distinct1  distinct2   entropy
blip_base_raw          550.893675  0.751951   0.075519   0.199601  5.773975
blip_large_raw          78.689783  0.876651   0.080680   0.236058  6.246690
blip_base_fine_tune   7845.087584  0.991387   0.164090   0.464716  6.574113
blip_large_fine_tune  6925.955081  0.989676   0.178941   0.494212  6.588131
bart_lora              167.422294  0.737324   0.131821   0.451995  8.649057
flan_aug                69.064656  0.976922   0.182974   0.399770  7.882467
